<div style="text-align: left; margin-bottom: 20px;">
  <img src="https://umd-brand.transforms.svdcdn.com/production/uploads/images/logos-primary.jpg?w=1801&h=601&auto=compress%2Cformat&fit=crop&dm=1613775207&s=71ce45031f9164cb409f11a2e28d8b8c" 
       alt="UMD Logo" style="max-width: 300px; height: auto;" />
</div>

# DATA/MSML 641: Natural Language Processing
## Session 7: Transformers

**University of Maryland, College Park**  
**Fall 2026**  
**Instructor**: Armin Mehrabian  
**Date**: October 20, 2026

<div style="text-align: left; margin-bottom: 20px;">
  <img src="https://umd-brand.transforms.svdcdn.com/production/uploads/images/logos-primary.jpg?w=1801&h=601&auto=compress%2Cformat&fit=crop&dm=1613775207&s=71ce45031f9164cb409f11a2e28d8b8c" 
       alt="UMD Logo" style="max-width: 300px; height: auto;" />
</div>

# Part 1: Introduction to Transformers

In this lecture, we will learn about **Transformers**, the architecture that powers modern Large Language Models (LLMs).

# LLMs are built out of transformers

**Transformer**: a specific kind of network architecture, like a fancier feedforward network, but based on attention.

- The foundational paper that introduced transformers
- Relies on self-attention


<div style="text-align: left; margin-bottom: 20px;">
  <img src="img/attention_is_all_you_need.png" 
       alt="Attention" style="height: auto;" />
</div>

# Part 1: Attention

Before diving into the full transformer architecture, let's understand the key mechanism that makes it work: **attention**.

# Problem with static embeddings (word2vec)

They are **static**! The embedding for a word doesn't reflect how its meaning changes in context.

Example:

> *The chicken didn't cross the road because **it** was too tired*

What is the meaning represented in the static embedding for "it"?

The static embedding cannot capture whether "it" refers to the chicken or the road - it's always the same vector regardless of context.

# Contextual Embeddings

**Key Intuition**: A representation of the meaning of a word should be different in different contexts!

**Contextual Embedding**: Each word has a different vector that expresses different meanings depending on the surrounding words.

**How to compute contextual embeddings?**
- **Attention**

# Contextual Embeddings: Example

Consider:

> *The chicken didn't cross the road because **it***

**What should be the properties of "it"?**

At this point in the sentence, it's probably referring to either the chicken or the road.

Compare:
- *The chicken didn't cross the road because it was too **tired*** → "it" refers to the chicken
- *The chicken didn't cross the road because it was too **wide*** → "it" refers to the road

# Intuition of attention

Build up the contextual embedding for a word by **selectively integrating information** from all the neighboring words.

We say that a word **"attends to"** some neighboring words more than others.

# Intuition of attention

<center>
<img width=60% src="img/attention_distrib.png">
</center>


The word "it" in layer k+1 is formed by attending to relevant words from layer k.

# Intuition of attention

<center>
<img width=80% src="img/self_attention_layer.png">
</center>

**Note:** This is a specific type of attention called, **causal (masked) self-attention**. Each position attends only to itself and earlier positions,

# Simplified attention

**Idea:** $\mathbf{a}_i$ = a weighted sum of the *previous* token embeddings, weighted by how similar each one is to $x_i$.

**Input:** token embeddings (each $x_j \in \mathbb{R}^d$)

$$x_1,\; x_2,\; \dots,\; x_{i-1},\; \underbrace{x_i}_{\text{current}}$$


**Step 1: score** (raw similarity, un-normalized)
$$\text{score}(x_i, x_j) = x_i \cdot x_j \qquad j \le i$$

**Step 2: normalize** (weights sum to 1)
$$\alpha_{ij} = \frac{\exp(\text{score}(x_i,x_j))}{\sum_{k \le i}\exp(\text{score}(x_i,x_k))}$$

**Step 3: mix**
$$\mathbf{a}_i = \sum_{j \le i} \alpha_{ij}\, x_j$$

**Note:** same $x_j$ plays three roles here

    1. query 
    2. key
    3. value 
    
Q/K/V comes next.

# An Actual Attention Head: slightly more complicated

High-level idea: instead of using vectors (like $x_i$ and $x_4$) directly, we'll represent 3 separate roles each vector $x_i$ plays:

- **query**: As *the current element* being compared to the preceding inputs
- **key**: As *a preceding input* that is being compared to the current element to determine a similarity
- **value**: A value of a preceding element that gets weighted and summed

<center>
<img width=80% src="img/attention_qv.png">
</center>

<center>
<img width=80% src="img/attention_q_k_v.png">
</center>

# An Actual Attention Head: Projection Matrices

We'll use matrices to project each vector $x_i$ into a representation of its role as query, key, value:

- **query**: $\mathbf{W}^Q$
- **key**: $\mathbf{W}^K$
- **value**: $\mathbf{W}^V$

$$\mathbf{q}_i = x_i \mathbf{W}^Q; \quad \mathbf{k}_i = x_i \mathbf{W}^K; \quad \mathbf{v}_i = x_i \mathbf{W}^V$$

# One attention head

$$\mathbf{q}_i = x_i \mathbf{W}^Q; \quad \mathbf{k}_j = x_j \mathbf{W}^K; \quad \mathbf{v}_j = x_j \mathbf{W}^V$$

$$\text{score}(x_i, x_j) = \frac{\mathbf{q}_i \cdot \mathbf{k}_j}{\sqrt{d_k}}$$

$$\alpha_{ij} = \text{softmax}(\text{score}(x_i, x_j)) \quad \forall j \leq i$$

$$\mathbf{a}_i = \sum_{j \leq i} \alpha_{ij} \mathbf{v}_j$$

In [1]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

def simple_attention(query, keys, values):
    d_k = keys.shape[-1]
    scores = (query @ keys.T) / np.sqrt(d_k)      # (1, 3)
    scores = scores - scores.max(axis=-1, keepdims=True)   # numerical stability
    weights = np.exp(scores)
    weights = weights / weights.sum(axis=-1, keepdims=True)
    output = weights @ values                      # (1, d)
    return output, weights

# Example usage
np.random.seed(0)
d = 4
query  = np.random.randn(1, d)
keys   = np.random.randn(3, d)
values = np.random.randn(3, d)

output, weights = simple_attention(query, keys, values)
print("Attention weights:", weights)
print("Weights sum to:", weights.sum())
print("Output shape:", output.shape)

Attention weights: [[0.38811428 0.36696907 0.24491666]]
Weights sum to: 1.0
Output shape: (1, 4)


# Example: Calculating the value of a3

<center>
<img width=80% src="img/a3_self_attention.png">
</center>

1. Generate key, query, value vectors from x1, x2, x3 using WK, WQ, WV
2. Compare x3's query with the keys for x1, x2, and x3 (dot products)
3. Divide scores by √dk
4. Turn into weights via softmax (α3,1, α3,2, α3,3)
5. Weigh each value vector
6. Sum the weighted value vectors to get a3


# Multi-head attention

Instead of one attention head, we'll have **lots of them**!

**Intuition**: Each head might be attending to the context for different purposes
- Different linguistic relationships or patterns in the context

$$\mathbf{q}_i^c = x_i \mathbf{W}^{Q_c}; \quad \mathbf{k}_j^c = x_j \mathbf{W}^{K_c}; \quad \mathbf{v}_j^c = x_j \mathbf{W}^{V_c}; \quad \forall c \quad 1 \leq c \leq h$$

$$\text{score}^c(x_i, x_j) = \frac{\mathbf{q}_i^c \cdot \mathbf{k}_j^c}{\sqrt{d_k}}$$

$$\alpha_{ij}^c = \text{softmax}(\text{score}^c(x_i, x_j)) \quad \forall j \leq i$$

$$\text{head}_i^c = \sum_{j \leq i} \alpha_{ij}^c \mathbf{v}_j^c$$

$$\mathbf{a}_i = (\text{head}_i^1 \oplus \text{head}_i^2 ... \oplus \text{head}_i^h) \mathbf{W}^O$$

$$\text{MultiHeadAttention}(x_i, [x_1, \cdots, x_N]) = \mathbf{a}_i$$

# Multi-head attention

<center>
<img width=80% src="img/multi_headed_attention.png">
</center>

# Summary of Attention

Attention is a method for enriching the representation of a token by incorporating contextual information.

**The result**: The embedding for each word will be different in different contexts!

**Contextual embeddings**: A representation of word meaning in its context.

We'll see in the next section that attention can also be viewed as a way to move information from one token to another.

<div style="text-align: left; margin-bottom: 20px;">
  <img src="https://umd-brand.transforms.svdcdn.com/production/uploads/images/logos-primary.jpg?w=1801&h=601&auto=compress%2Cformat&fit=crop&dm=1613775207&s=71ce45031f9164cb409f11a2e28d8b8c" 
       alt="UMD Logo" style="max-width: 300px; height: auto;" />
</div>

# Part 2: The Transformer Architecture

Now let's see how attention is integrated into transformer blocks that form the building blocks of modern LLMs.

<center>
<img width=80% src="img/transformer-block.png">
</center>

# Encoder vs. Decoder: what differs

| | **Encoder** (left tower) | **Decoder** (right tower) |
|---|---|---|
| Sees | all positions, $\forall j$ | only $j \le i$ (masked) |
| Attention sublayers | 1: self-attention | 2: masked self + **cross** |
| Input | source sequence | target, shifted right |
| Output used as | contextual representations | probabilities over vocab |

**Cross-attention** is the only new machinery: $\mathbf{q}$ from the decoder, $\mathbf{k}$ and $\mathbf{v}$ from the encoder's final output. Same equations, K/V just come from the other tower.

Everything else is identical on both sides: multi-head attention, feedforward, Add & Norm, positional encoding. On the self-attention side, the mask is the entire difference. Same code, one boolean argument.

# Why Q from the decoder, K/V from the encoder

Recall the roles:

- **query** = what I am looking for
- **key** = what I advertise having
- **value** = what I hand over if selected

Generating the French word after "Le chat":

- The decoder knows what it still **needs**. It has a determiner and a noun, so its query says *I need the verb, and I need to know what the cat did*.
- The encoder holds the **content**. Each English position advertises what it is, "sat" advertises *past-tense verb of sitting*, and hands over its meaning if picked.

The need lives on the decoder side, the content lives on the encoder side.

# Consequences

**Information flows one way.** Encoder to decoder only. The encoder never sees the target.

**The encoder runs once.** Its output is computed a single time and reused at every decoding step, so K and V can be cached for the whole generation.

**Two questions, two sublayers:**

| sublayer | asks |
|---|---|
| masked self-attention | what have I already said? |
| cross-attention | what was I given? |

That is why the decoder needs both.

# The encoder runs once per input

Translating *"The cat sat on the mat"* into French, producing 8 output tokens:

| | runs | why |
|---|---|---|
| Encoder | **1×** | source never changes while decoding |
| Decoder | **8×** | one pass per generated token |

Step 1. Encode the source, get 6 vectors. Compute $K, V$ for each cross-attention layer. Cache.
Step 2. Generate *Le*, *chat*, *s'est*, ... Every step re-reads the **same** cached $K, V$.

Nothing about generating token 7 changes what "cat" means in the input.

<center>
<img width=80% src="img/transformer_language_model.png">
</center>

# Feedforward Layer (FFN)

Transformers also need **nonlinearities**  -  this is where the feedforward layer comes in.

$$\text{FFN}(x_i) = \text{ReLU}(x_i \mathbf{W}_1 + b_1)\mathbf{W}_2 + b_2$$

> **Note**: Modern transformers (Llama, Mistral, etc.) typically replace ReLU with **SwiGLU** or **GELU** activations, which empirically improve performance. The SwiGLU variant uses a gated linear unit:
> $$\text{FFN}_{\text{SwiGLU}}(x) = (x\mathbf{W}_1 \odot \text{SiLU}(x\mathbf{W}_{\text{gate}}))\mathbf{W}_2$$

- Can you identify Residual paths?
- Residual paths vs no-residual path?
- What do they imply?

<center>
<img width=80% src="img/transformer_full.png">
</center>

# The residual stream: each token gets passed up and modified

<center>
<img width=80% src="img/residual_stream.png">
</center>


- Input xi at bottom
- Flows up through hi-1, hi, hi+1
- Two modification points marked with ⊕:
  1. After Layer Norm → MultiHead Attention
  2. After Layer Norm → Feedforward
- Both modifications are added back to the residual stream]


Each token has its own **residual stream** that flows upward through the network, getting modified at each layer.

# The vector xi is normalized twice

<center>
<img width=80% src="img/residual_stream.png">
</center>


Same residual stream diagram highlighting both Layer Norm components (before attention and before feedforward)

# Layer Norm

Layer norm is a variation of the z-score from statistics, applied to a single vector in a hidden layer.

$$\mu = \frac{1}{d} \sum_{i=1}^{d} x_i$$
$$\sigma = \sqrt{\frac{1}{d} \sum_{i=1}^{d} (x_i - \mu)^2}$$
$$\hat{x} = \frac{(x - \mu)}{\sigma}$$
$$\text{LayerNorm}(x) = \gamma \frac{(x - \mu)}{\sigma} + \beta$$

where $\gamma$ and $\beta$ are learnable parameters.

**Why:**
- Keeps activations at a consistent scale as they pass through many layers, so nothing blows up or vanishes with depth.
- Stabilizes gradients, which is what lets deep stacks train at all.

# Where does the norm go?

**Post-norm** (2017 paper, the "Add & Norm" boxes in the figure):
$$x_{out} = \text{LayerNorm}(x + \text{Sublayer}(x))$$

**Pre-norm** (everything since roughly 2020):
$$x_{out} = x + \text{Sublayer}(\text{LayerNorm}(x))$$

The difference: in pre-norm the residual path runs clean from input to output with no normalization on it, so gradients reach early layers unattenuated. In post-norm the norm sits on the main path, and deep stacks either fail to converge or need learning-rate warmup to survive early training.

GPT-2 switched to pre-norm and everyone followed. Llama, GPT-3, and essentially all current models are pre-norm.

In [3]:
# Layer Normalization implementation
def layer_norm(x, gamma, beta, eps=1e-5):
    """
    Apply layer normalization.
    
    Args:
        x: Input vector
        gamma: Scale parameter
        beta: Shift parameter
        eps: Small constant for numerical stability
    
    Returns:
        Normalized vector
    """
    mean = np.mean(x)
    var = np.var(x)
    x_norm = (x - mean) / np.sqrt(var + eps)
    return gamma * x_norm + beta

# Example
x = np.random.randn(10)
gamma = np.ones(10)
beta = np.zeros(10)

normalized = layer_norm(x, gamma, beta)
print(f"Original mean: {np.mean(x):.4f}, std: {np.std(x):.4f}")
print(f"Normalized mean: {np.mean(normalized):.4f}, std: {np.std(normalized):.4f}")

Original mean: 0.2908, std: 1.0792
Normalized mean: 0.0000, std: 1.0000


# Putting together a single transformer block


$$\mathbf{t}_i^1 = \text{LayerNorm}(x_i)$$

$$\mathbf{t}_i^2 = \text{MultiHeadAttention}(\mathbf{t}_i^1, [x_1^1, \cdots, x_N^1])$$

$$\mathbf{t}_i^3 = \mathbf{t}_i^2 + x_i$$

$$\mathbf{t}_i^4 = \text{LayerNorm}(\mathbf{t}_i^3)$$

$$\mathbf{t}_i^5 = \text{FFN}(\mathbf{t}_i^4)$$

$$\mathbf{h}_i = \mathbf{t}_i^5 + \mathbf{t}_i^3$$

<center>
<img width=80% src="img/residual_stream.png">
</center>

# A transformer is a stack of these blocks

All the vectors are of the same dimensionality $d$


<center>
<img width=80% src="img/2_attention_blocks.png">
</center>

Two transformer blocks stacked vertically (Block 1 and Block 2), showing the same internal structure for each block


# Residual streams and attention

Notice that all parts of the transformer block apply to **1 residual stream** (1 token).

**Except attention**, which takes information from other tokens.

Elhage et al. (2021) show that we can view attention heads as literally **moving information** from the residual stream of a neighboring token into the current stream.

<center>
<img width=80% src="img/residual_streams_attention.png">
</center>

- Multiple vertical bars representing residual streams for different tokens
- Arrow showing information flowing from Token A residual stream to Token B residual stream via attention]


<div style="text-align: left; margin-bottom: 20px;">
  <img src="https://umd-brand.transforms.svdcdn.com/production/uploads/images/logos-primary.jpg?w=1801&h=601&auto=compress%2Cformat&fit=crop&dm=1613775207&s=71ce45031f9164cb409f11a2e28d8b8c" 
       alt="UMD Logo" style="max-width: 300px; height: auto;" />
</div>

# Part 3: Parallelizing Attention Computation

One of the key advantages of transformers is the ability to parallelize computation efficiently.

# Parallelizing computation using X

For attention/transformer block we've been computing a **single** output at a **single** time step $i$ in a **single** residual stream.

But we can pack the $N$ tokens of the input sequence into a single matrix $\mathbf{X}$ of shape $[N \times d]$.

- Each row of $\mathbf{X}$ is the embedding of one token of the input
- $\mathbf{X}$ can have 1K-32K rows, each of the dimensionality of the embedding $d$ (the model dimension)

$$\mathbf{Q} = \mathbf{X}\mathbf{W}^Q; \quad \mathbf{K} = \mathbf{X}\mathbf{W}^K; \quad \mathbf{V} = \mathbf{X}\mathbf{W}^V$$

# QKᵀ

Now we can do a single matrix multiply to combine $\mathbf{Q}$ and $\mathbf{K}^T$


<center>
<img width=80% src="img/QK-matrix.png">
</center>

- Rows labeled q1, q2, q3, q4
- Columns labeled k1, k2, k3, k4
- Each cell contains the dot product q·k (e.g., q1·k1, q1·k2, etc.)]


This $N \times N$ matrix contains all pairwise query-key similarities.

# Parallelizing attention

Scale the scores, take the softmax, and then multiply the result by $\mathbf{V}$ resulting in a matrix of shape $N \times d$

- An attention vector for each input token

$$\mathbf{A} = \text{softmax}\left(\text{mask}\left(\frac{\mathbf{Q}\mathbf{K}^T}{\sqrt{d_k}}\right)\right) \mathbf{V}$$

# Masking out the future

$$\mathbf{A} = \text{softmax}\left(\text{mask}\left(\frac{\mathbf{Q}\mathbf{K}^T}{\sqrt{d_k}}\right)\right) \mathbf{V}$$

**What is this mask function?**

$\mathbf{Q}\mathbf{K}^T$ has a score for each query dot every key, *including those that follow the query*.

Guessing the next word is pretty simple if you already know it!

# Masking out the future

Add $-\infty$ to cells in upper triangle. The softmax will turn it to 0.


- Lower triangle filled with actual values (q1·k1, q2·k1, q2·k2, etc.)
- Upper triangle filled with -∞

<center>
<img width=80% src="img/masked-QK-matrix.png">
</center>

In [4]:
# Causal mask demonstration
def create_causal_mask(size):
    """
    Create a causal mask for attention.
    
    Args:
        size: Size of the mask (number of tokens)
    
    Returns:
        Mask matrix where upper triangle is -inf
    """
    mask = np.triu(np.ones((size, size)) * -np.inf, k=1)
    return mask

# Example
mask = create_causal_mask(5)
print("Causal mask:")
print(mask)

# Apply to scores
scores = np.random.randn(5, 5)
masked_scores = scores + mask

print("\nScores after masking:")
print(masked_scores)

Causal mask:
[[  0. -inf -inf -inf -inf]
 [  0.   0. -inf -inf -inf]
 [  0.   0.   0. -inf -inf]
 [  0.   0.   0.   0. -inf]
 [  0.   0.   0.   0.   0.]]

Scores after masking:
[[-0.38732682        -inf        -inf        -inf        -inf]
 [ 1.9507754  -0.50965218        -inf        -inf        -inf]
 [-1.61389785 -0.21274028 -0.89546656        -inf        -inf]
 [-1.18063218 -0.02818223  0.42833187  0.06651722        -inf]
 [-0.63432209 -0.36274117 -0.67246045 -0.35955316 -0.81314628]]


# Another point: Attention is quadratic in length

$$\mathbf{A} = \text{softmax}\left(\text{mask}\left(\frac{\mathbf{Q}\mathbf{K}^T}{\sqrt{d_k}}\right)\right) \mathbf{V}$$

The $\mathbf{Q}\mathbf{K}^T$ matrix has shape $N \times N$, where $N$ is the sequence length.

This means:
- Memory requirements scale as $O(N^2)$
- Computation scales as $O(N^2)$

This is why context length is a major bottleneck for transformers!

**How modern systems address this:**
- **Flash Attention** (Dao et al., 2022): Restructures computation to minimize GPU memory reads/writes  -  computes *exact* attention 2-4x faster without ever materializing the full N×N matrix in memory
- **Sliding Window Attention**: Each token only attends to a local window (e.g., 4096 tokens); information propagates through stacked layers
- **Grouped-Query Attention (GQA)**: Shares K/V heads across query heads, reducing the KV cache memory bottleneck at inference

# Attention again: complete picture

<center>
<img width=80% src="img/masked-full-attention.png">
</center>

- Input X (N×d) split into three paths
- Multiply by WQ (d×dk), WK (d×dk), WV (d×dv) to get Q, K, V
- Q (N×dk) × Kᵀ (dk×N) = QKᵀ (N×N)
- Apply mask to upper triangle
- Softmax to get attention weights
- Multiply by V (N×dv) to get A (N×dv)]


# Parallelizing Multi-head Attention

$$\mathbf{Q}^i = \mathbf{X}\mathbf{W}^{Q_i}; \quad \mathbf{K}^i = \mathbf{X}\mathbf{W}^{K_i}; \quad \mathbf{V}^i = \mathbf{X}\mathbf{W}^{V_i}$$

$$\text{head}_i = \text{SelfAttention}(\mathbf{Q}^i, \mathbf{K}^i, \mathbf{V}^i) = \text{softmax}\left(\frac{\mathbf{Q}^i {\mathbf{K}^i}^T}{\sqrt{d_k}}\right) \mathbf{V}^i$$

$$\text{MultiHeadAttention}(\mathbf{X}) = (\text{head}_1 \oplus \text{head}_2 ... \oplus \text{head}_h)\mathbf{W}^O$$

# Parallelizing Multi-head Attention: Complete Block

> **Note on normalization order**: The equations below show **post-norm** (LayerNorm after residual addition), which was used in the original "Attention Is All You Need" paper. Most modern transformers use **pre-norm** (LayerNorm *before* attention/FFN, as shown in the single-block slide earlier), which improves training stability.

Matrix form of one transformer block (post-norm variant):

$$\mathbf{T}^1 = \text{MultiHeadAttention}(\mathbf{X})$$

$$\mathbf{T}^2 = \mathbf{X} + \mathbf{T}^1$$

$$\mathbf{T}^3 = \text{LayerNorm}(\mathbf{T}^2)$$

$$\mathbf{T}^4 = \text{FFN}(\mathbf{T}^3)$$

$$\mathbf{T}^5 = \mathbf{T}^4 + \mathbf{T}^3$$

$$\mathbf{H} = \text{LayerNorm}(\mathbf{T}^5)$$

In [5]:
# Simplified multi-head attention implementation
def multi_head_attention(X, W_Q, W_K, W_V, W_O, num_heads, d_k):
    """
    Multi-head attention mechanism.
    
    Args:
        X: Input matrix (N x d)
        W_Q, W_K, W_V: Weight matrices for queries, keys, values
        W_O: Output projection matrix
        num_heads: Number of attention heads
        d_k: Dimension of each head
    
    Returns:
        Output after multi-head attention
    """
    N, d = X.shape
    
    # Compute Q, K, V for all heads
    Q = np.dot(X, W_Q)
    K = np.dot(X, W_K)
    V = np.dot(X, W_V)
    
    # Reshape for multiple heads
    Q = Q.reshape(N, num_heads, d_k)
    K = K.reshape(N, num_heads, d_k)
    V = V.reshape(N, num_heads, d_k)
    
    # Compute attention for each head
    heads = []
    for h in range(num_heads):
        # Attention scores
        scores = np.dot(Q[:, h, :], K[:, h, :].T) / np.sqrt(d_k)
        
        # Apply causal mask
        mask = create_causal_mask(N)
        scores = scores + mask
        
        # Softmax (numerically stable)
        scores_shifted = scores - np.max(scores, axis=-1, keepdims=True)
        attention_weights = np.exp(scores_shifted) / np.sum(np.exp(scores_shifted), axis=-1, keepdims=True)
        
        # Apply attention to values
        head_output = np.dot(attention_weights, V[:, h, :])
        heads.append(head_output)
    
    # Concatenate heads
    concat_heads = np.concatenate(heads, axis=-1)
    
    # Final projection
    output = np.dot(concat_heads, W_O)
    
    return output

# Example usage
N, d = 4, 8
num_heads = 2
d_k = d // num_heads

X = np.random.randn(N, d)
W_Q = np.random.randn(d, num_heads * d_k)
W_K = np.random.randn(d, num_heads * d_k)
W_V = np.random.randn(d, num_heads * d_k)
W_O = np.random.randn(num_heads * d_k, d)

output = multi_head_attention(X, W_Q, W_K, W_V, W_O, num_heads, d_k)
print(f"Input shape: {X.shape}")
print(f"Output shape: {output.shape}")

Input shape: (4, 8)
Output shape: (4, 8)


<div style="text-align: left; margin-bottom: 20px;">
  <img src="https://umd-brand.transforms.svdcdn.com/production/uploads/images/logos-primary.jpg?w=1801&h=601&auto=compress%2Cformat&fit=crop&dm=1613775207&s=71ce45031f9164cb409f11a2e28d8b8c" 
       alt="UMD Logo" style="max-width: 300px; height: auto;" />
</div>

# Part 4: Input and Output - Position Embeddings and the Language Model Head

Now let's understand how inputs are encoded and outputs are generated in transformers.

# What is a token, really?

> A **token** is one element of the sequence the transformer operates over, represented as a vector.

Tokenization is just the choice of *how to cut the input* into such elements.
Nothing in the attention equations references language.

| Domain | One token = | Sequence length |
|---|---|---|
| Text | subword (`" chick"`, `"en"`) | ~1.3 × words |
| Image (ViT) | 16×16 pixel patch | 196 for 224×224 |
| Audio | ~25 ms spectrogram frame | 50 / second |
| Video | spacetime patch | patches × frames |
| Proteins | one amino acid | length of chain |


# Position Embeddings: The Problem

Attention is **permutation invariant**. It computes weighted sums over values, order plays no role.

**"dog bites man"** and **"man bites dog"** produce the *same set* of token interactions unless something tells the model which token came first.

**Goal**: inject order information into the representation, somehow.

# Token and Position Embeddings in Text

Each row of $\mathbf{X}$ (shape $[N \times d]$) is one token's representation in context.

It's built by **adding** two $d$-dimensional vectors:

$$\mathbf{x}_i = \mathbf{e}_{\text{token}_i} + \mathbf{e}_{\text{pos}_i}$$

- **Token embedding**: *what* word is this?
- **Position embedding**: *where* is it in the sequence?

Note: addition, not concatenation. It's a design choice (parameter-efficient), not a necessity.

## Token Embeddings

Embedding matrix $\mathbf{E}$ has shape $[|V| \times d]$:

- One row per vocabulary token
- Each row is that token's $d$-dimensional vector

**Example**: `"Thanks for all the"`

1. BPE tokenize → vocab indices: $w = [5, 4000, 10532, 2224]$
2. Look up rows in $\mathbf{E}$: `(row 5, row 4000, row 10532, row 2224)`

That's it  -  embedding is just a lookup table.

# Two Design Axes

Every position embedding scheme is a choice along two independent axes:

**Deterministic vs. Learned**
- Deterministic: fixed formula, no parameters, no training (sinusoidal, RoPE, ALiBi)
- Learned: initialized randomly, updated during training (absolute lookup table)

**Absolute vs. Relative**
- Absolute: encodes *this token's* position in the sequence (learned lookup, sinusoidal)
- Relative: encodes the *distance between* two tokens (RoPE, ALiBi)

Most of the historical development in position embeddings is really just moving through these four combinations.

## Position Embeddings


<center>
<img width=80% src="img/word-position-embedding.png">
</center>

# Deterministic Absolute: Sinusoidal

Same absolute-position idea, but **no learning**. Position $i$ maps to a fixed vector via sine/cosine functions of different frequencies.

- Defined for *any* position, no max length baked in
- No parameters, nothing for training to get wrong
- Used in the original Transformer (Vaswani et al., 2017)

Still absolute: two tokens 3 apart get different treatment depending on *where* they are in the sequence, not just their distance.

# Deterministic Relative: RoPE and ALiBi

The modern move: switch to **relative** position, while staying **deterministic**.

**RoPE**: rotate $q$/$k$ vectors by an angle proportional to position.
- Each position gets its own rotation angle, applied inside the attention computation, not added to the embedding
- Rotating both $q_m$ and $k_n$ means their dot product depends only on $(m - n)$, the *gap* between them, not on $m$ and $n$ individually
- Two tokens 3 apart always interact the same way, whether they're at positions (1, 4) or (50, 53)

**ALiBi**: skip embeddings altogether, bias the attention scores directly.
- Attention score becomes $q_i \cdot k_j - m \cdot (i - j)$
- The penalty term grows with distance, so farther tokens get pulled down before the softmax, a built-in "prefer nearby context" bias
- $m$ is a fixed, head-specific slope, not learned

Both give the model relative distance directly, without a learned lookup table and without needing an absolute position at all.

# Learned Absolute Position Embeddings

The simplest point in the design space: **learned + absolute**.

$\mathbf{E}_{pos}$ has shape $[N_{max} \times d]$, one row per position, just like $\mathbf{E}$ has one row per token.

$N_{max}$ is the maximum sequence length fixed at training time, i.e. the model's context window size.

- Randomly initialized, then trained like any other parameter
- Position 3 gets a vector, the same way token "fish" does

**Example**: "Thanks for all the" → look up rows $(0, 1, 2, 3)$, add to the token embeddings.

**Limitation**: no row exists past $N_{max}$, so the model can't handle unseen lengths.

# Learned Relative Position Embeddings

Fill the fourth quadrant: **relative** distance, but **learned** rather than fixed.

- Learn an embedding (or scalar bias) per relative offset $(i - j)$, instead of per absolute position
- **Shaw et al. (2018)**: learned vector added into the attention computation, indexed by offset
- **T5 (2020)**: learned scalar bias per relative-distance bucket, added to attention scores, similar shape to ALiBi's penalty but learned instead of fixed

Historically a stepping stone: 
- relative position helped before RoPE existed, but learning a separate parameter per offset still doesn't generalize past the offsets seen in training
- which is exactly the gap RoPE and ALiBi close by making the relative signal a *fixed formula* instead of a *learned lookup*.

# Byte Pair Encoding (BPE)

**Goal.** Build a subword vocabulary so any text can be tokenized (few OOVs) with a manageable vocab size.

**Training (learn merges once):**
1. Start with characters (plus an end-of-word marker like `</w>`).
2. Count all adjacent symbol pairs in a large corpus.
3. Merge the most frequent pair into a new symbol; add it to the vocab.
4. Repeat steps 2–3 for \(M\) merges → final vocab size \(|V| = |\text{chars}| + M\).

**Tokenization at inference (apply learned merges):**
For each word, repeatedly apply the learned merges (most-specific first) until no merge matches.


## Example: “Thanks for all the”

*(Toy illustration of merges; exact steps depend on the learned merge list.)*

- `Thanks</w>`  
  `T h a n k s </w>` → `Th a n k s </w>` → `Tha nk s </w>` → `Than ks </w>` → `Thanks </w>` → `Thanks`
- `for</w>`  
  `f o r </w>` → `fo r </w>` → `for </w>` → `for`
- `all</w>`  
  `a l l </w>` → `al l </w>` → `all`
- `the</w>`  
  `t h e </w>` → `the`

**Resulting tokens:** `["Thanks", "for", "all", "the"]`

**Map to vocab ids (from previous slide):**  
w = [5, 4000, 10532, 2224]



In [6]:
# Position embedding demonstration
def create_positional_embeddings(max_len, d_model):
    """
    Create learnable positional embeddings.
    
    Args:
        max_len: Maximum sequence length
        d_model: Model dimension
    
    Returns:
        Position embedding matrix
    """
    # Initialize randomly (in practice, these are learned)
    pos_embeddings = np.random.randn(max_len, d_model) * 0.02
    return pos_embeddings

# Example: Combine token and position embeddings
vocab_size = 50000
d_model = 512
max_len = 1024

# Token embedding matrix
token_embeddings = np.random.randn(vocab_size, d_model) * 0.02

# Position embedding matrix
pos_embeddings = create_positional_embeddings(max_len, d_model)

# Example sentence: [5, 4000, 10532, 2224]
tokens = [5, 4000, 10532, 2224]
seq_len = len(tokens)

# Get token embeddings
token_emb = token_embeddings[tokens]

# Get position embeddings
position_emb = pos_embeddings[:seq_len]

# Combine
input_embeddings = token_emb + position_emb

print(f"Token embeddings shape: {token_emb.shape}")
print(f"Position embeddings shape: {position_emb.shape}")
print(f"Combined input embeddings shape: {input_embeddings.shape}")

Token embeddings shape: (4, 512)
Position embeddings shape: (4, 512)
Combined input embeddings shape: (4, 512)


# Language modeling head: Unembedding layer

**Unembedding layer**: Linear layer projects from $\mathbf{h}_N^L$ (shape $[1 \times d]$) to logit vector

**Why "unembedding"?** Tied to $\mathbf{E}^T$

**Weight tying**: We use the same weights for two different matrices

Unembedding layer maps from an embedding to a $1 \times |V|$ vector of logits

- Embedding matrix $\mathbf{E}$ (shape $[|V| \times d]$): maps from vocabulary to embedding
- Unembedding matrix $\mathbf{E}^T$ (shape $[d \times |V|]$): maps from embedding back to vocabulary


<center>
<img width=80% src="img/lm-unembedding.png">
</center>

# Where this goes next

Since 2017, almost everything around self-attention has been changed: grouped-query attention, Flash Attention, RMSNorm, SwiGLU, Mixture of Experts, and more.
Running these models fast is its own field: prefill and decode, the KV cache, quantization, batching, speculative decoding.

These are covered as a map in **Session 12: Frontier NLP**.


# References and Credits

This lecture is based on materials from:

**Primary Source**:
Daniel Jurafsky and James H. Martin. 2025. *Speech and Language Processing: An Introduction to Natural Language Processing, Computational Linguistics, and Speech Recognition with Language Models*, 3rd edition. Online manuscript released August 24, 2025. https://web.stanford.edu/~jurafsky/slp3.

**Original Transformer Paper**:
Vaswani, A., Shazeer, N., Parmar, N., Uszkoreit, J., Jones, L., Gomez, A. N., Kaiser, Ł., & Polosukhin, I. (2017). *Attention is all you need*. In Advances in Neural Information Processing Systems (pp. 5998-6008).

**Additional References**:
- Elhage, N., et al. (2021). *A Mathematical Framework for Transformer Circuits*. Anthropic.
- Kaplan, J., et al. (2020). *Scaling Laws for Neural Language Models*. arXiv:2001.08361.
- Hu, E. J., et al. (2021). *LoRA: Low-Rank Adaptation of Large Language Models*. arXiv:2106.09685.
